# Seeded Run 1 — 4-Class Task

This notebook trains five matched seeded pairs for the 4-class SHD parity×language task.

Each training cell handles one seed pair independently. Once a pair finishes, its checkpoints and summaries are already saved, so you can stop there and later continue with the next seed cell instead of rerunning a full loop.

Cell 2 defines the seeds used in this notebook. Cell 3 verifies three things before training starts:
1. this notebook is locked to the 4-class parity×language classification task
2. the homogeneous and heterogeneous model definitions are set up correctly
3. the log-normal hidden tau_m samples vary across the configured seeds

In [1]:
import sys
sys.path.insert(0, r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files")

from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

import seeded_runs_common as seeded_runs_common

seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

RUN_LABEL = "seeded_run_1"
TASK_KEY = "4class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
MASTER_SEEDS = [101, 202, 210, 340, 440]

RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
RESULT_STEM = RUN_DIR / f"{RUN_LABEL}_checkpoint_summary"
PAIR_STEM = RUN_DIR / f"{RUN_LABEL}_pair_summary"
PAIRED_ACC_CSV = RUN_DIR / f"{RUN_LABEL}_paired_accuracy.csv"

TASK_DEF = TASKS[TASK_KEY]

print(f"Device: {DEVICE}")
print(f"Train file exists: {SHD_TRAIN.exists()}")
print(f"Test file exists: {SHD_TEST.exists()}")
print(f"Tau artifact exists: {TAU_ARTIFACT_PATH.exists()}")
print(f"Run directory: {RUN_DIR}")
print(f"Master seeds: {MASTER_SEEDS}")
print(f"Task name: {TASK_DEF['task_name']}")
print(f"Task outputs: {TASK_DEF['nb_outputs']}")

Device: cuda
Train file exists: True
Test file exists: True
Tau artifact exists: True
Run directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1\4class_lognormal
Master seeds: [101, 202, 210, 340, 440]
Task name: 4class_parity_language
Task outputs: 4


In [2]:
assert TASK_KEY == "4class"
assert TASK_DEF["nb_outputs"] == 4
assert TASK_DEF["task_name"] == "4class_parity_language"

# Verify 4-class parity×language label map (English/German × even/odd)
expected_4class_map = {}
for i in range(20):
    is_german = int(i >= 10)
    is_odd = int(i % 2 == 1)
    expected_4class_map[i] = is_german * 2 + is_odd
assert TASK_DEF["task_label_map"] == expected_4class_map

preview = build_seeded_pair(
    master_seed=MASTER_SEEDS[0],
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
preview_meta = preview["metadata"]

assert preview["hom_prms"]["nb_outputs"] == 4
assert preview["hetero_prms"]["nb_outputs"] == 4
assert not preview["hom_model"].network[0].alpha.requires_grad
assert not preview["hom_model"].network[0].beta.requires_grad
assert not preview["hetero_model"].network[0].alpha.requires_grad
assert not preview["hetero_model"].network[0].beta.requires_grad
assert preview_meta["linear_sync_verified"]
assert preview_meta["hom_hidden_tau_unique"] == 1
assert preview_meta["hetero_hidden_tau_unique"] > 1

sampling_rows = build_sampling_preview_rows(
    MASTER_SEEDS,
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
sampling_df = pd.DataFrame(sampling_rows).sort_values("pair_seed").reset_index(drop=True)

assert not sampling_df["sample_matches_previous"].any()

display(sampling_df)
print("4-class parity×language task mapping verified.")
print("Homogeneous anchor and heterogeneous sampled definitions verified.")
print("Log-normal hidden tau_m sampling varies across the configured seeds.")

,pair_seed,task_key,task_name,mem_distribution_family,linear_sync_verified,hetero_hidden_tau_unique,hom_hidden_tau_unique,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,sample_matches_previous
0,101,4class,4class_parity_language,lognormal,True,30,1,24.687252,3.410472,99.749887,37.499115,30.195999,False
1,202,4class,4class_parity_language,lognormal,True,28,1,29.308363,6.953492,99.749887,39.884979,30.480470,False
2,210,4class,4class_parity_language,lognormal,True,31,1,22.405558,3.855315,99.749887,34.397371,29.713145,False
3,340,4class,4class_parity_language,lognormal,True,31,1,23.273864,4.656051,99.749887,35.008043,30.389582,False
4,440,4class,4class_parity_language,lognormal,True,28,1,28.874491,5.547958,99.749887,41.448743,31.915950,False


4-class parity×language task mapping verified.
Homogeneous anchor and heterogeneous sampled definitions verified.
Log-normal hidden tau_m sampling varies across the configured seeds.


In [3]:
train_cache, test_cache = load_default_caches()

Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5


In [7]:
# ── 4-Class Training Optimizations ──────────────────────────────────
# The 4-class task is harder than 2-class and benefits from:
#   1. More epochs (50 vs default 25) — gives models time to converge
#   2. Weight decay (1e-4) — reduces overfitting seen in seed 101 homo
#   3. LR scheduler (ReduceLROnPlateau, patience=8, factor=0.5) — escapes plateaus
#   4. cuDNN benchmark — faster conv kernels on fixed-size inputs

import torch
import importlib

# Reload common module to pick up the LR scheduler changes
seeded_runs_common = importlib.reload(seeded_runs_common)

# Re-bind all the names we use (reload invalidates old references)
CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

# Speed: enable cuDNN auto-tuner for fixed-shape inputs
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# Override BASE_PRMS for the 4-class task
# These are picked up by build_seeded_pair → run_pair_training
seeded_runs_common.BASE_PRMS["nb_epochs"] = 50          # was 25
seeded_runs_common.BASE_PRMS["weight_decay"] = 1e-4     # was 0.0
seeded_runs_common.BASE_PRMS["lr_patience"] = 8          # scheduler: wait 8 epochs
seeded_runs_common.BASE_PRMS["lr_factor"] = 0.5          # scheduler: halve LR
seeded_runs_common.BASE_PRMS["lr_min"] = 5e-5            # scheduler: floor

print("4-class training optimizations applied:")
print(f"  nb_epochs   = {seeded_runs_common.BASE_PRMS['nb_epochs']}")
print(f"  weight_decay = {seeded_runs_common.BASE_PRMS['weight_decay']}")
print(f"  LR scheduler = ReduceLROnPlateau(patience=8, factor=0.5)")
print(f"  cuDNN benchmark = {torch.backends.cudnn.benchmark if torch.cuda.is_available() else 'N/A'}")
print("  Module reloaded — LR scheduler code is now active.")


4-class training optimizations applied:
  nb_epochs   = 50
  weight_decay = 0.0001
  LR scheduler = ReduceLROnPlateau(patience=8, factor=0.5)
  cuDNN benchmark = True
  Module reloaded — LR scheduler code is now active.


In [4]:
CHECKPOINT_KEY_FIELDS = ["pair_seed", "pair_role"]
PAIR_KEY_FIELDS = ["pair_seed"]

def show_run_status():
    checkpoint_rows = read_manifest_rows(RESULT_STEM)
    pair_rows = read_manifest_rows(PAIR_STEM)

    checkpoint_df = pd.DataFrame(checkpoint_rows)
    pair_df = pd.DataFrame(pair_rows)

    if not checkpoint_df.empty:
        checkpoint_df = checkpoint_df.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    if not pair_df.empty:
        pair_df = pair_df.sort_values("pair_seed").reset_index(drop=True)

    paired_acc_df = pd.DataFrame()
    if not checkpoint_df.empty:
        paired_acc_df = (
            checkpoint_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
            .reset_index()
            .sort_values("pair_seed")
            .reset_index(drop=True)
        )
        if {"heterogeneous_sampled", "homogeneous_anchor"}.issubset(paired_acc_df.columns):
            paired_acc_df["hetero_minus_hom"] = (
                paired_acc_df["heterogeneous_sampled"] - paired_acc_df["homogeneous_anchor"]
            )
            paired_acc_df.to_csv(PAIRED_ACC_CSV, index=False)

    return pair_df, checkpoint_df, paired_acc_df

def train_one_seed(master_seed):
    run_rows, pair_meta = run_pair_training(
        master_seed=master_seed,
        train_cache=train_cache,
        test_cache=test_cache,
        task_key=TASK_KEY,
        mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
        run_label=RUN_LABEL,
        skip_existing=True,
    )

    checkpoint_rows = upsert_rows(
        read_manifest_rows(RESULT_STEM),
        run_rows,
        CHECKPOINT_KEY_FIELDS,
    )
    pair_rows = upsert_rows(
        read_manifest_rows(PAIR_STEM),
        [build_pair_summary_row(pair_meta)],
        PAIR_KEY_FIELDS,
    )

    write_manifest_rows(checkpoint_rows, RESULT_STEM)
    write_manifest_rows(pair_rows, PAIR_STEM)

    pair_df, checkpoint_df, paired_acc_df = show_run_status()
    seed_df = checkpoint_df[checkpoint_df["pair_seed"] == master_seed].reset_index(drop=True)
    return seed_df, pair_df, checkpoint_df, paired_acc_df

print("Run helpers ready.")
print("Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.")

Run helpers ready.
Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.


In [5]:
# Train or reuse seed pair 101
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(101)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 101].reset_index(drop=True))


Seed 101 :: training homogeneous_anchor...
  epoch=001  train_acc=0.328  test_acc=0.399  (4.3 min)
  epoch=002  train_acc=0.414  test_acc=0.413  (3.3 min)
  epoch=003  train_acc=0.433  test_acc=0.402  (3.4 min)
  epoch=004  train_acc=0.466  test_acc=0.435  (3.4 min)
  epoch=005  train_acc=0.477  test_acc=0.446  (3.3 min)
  epoch=006  train_acc=0.507  test_acc=0.453  (3.2 min)
  epoch=007  train_acc=0.532  test_acc=0.502  (3.3 min)
  epoch=008  train_acc=0.565  test_acc=0.545  (3.4 min)
  epoch=009  train_acc=0.590  test_acc=0.536  (3.7 min)
  epoch=010  train_acc=0.600  test_acc=0.547  (3.5 min)
  epoch=011  train_acc=0.627  test_acc=0.585  (3.5 min)
  epoch=012  train_acc=0.646  test_acc=0.572  (3.5 min)
  epoch=013  train_acc=0.655  test_acc=0.566  (3.6 min)
  epoch=014  train_acc=0.683  test_acc=0.600  (3.7 min)
  epoch=015  train_acc=0.695  test_acc=0.596  (3.4 min)
  epoch=016  train_acc=0.718  test_acc=0.600  (3.3 min)
  epoch=017  train_acc=0.725  test_acc=0.596  (3.2 min)
  ep

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,101,heterogeneous_sampled,0.494700,1.190044,5882.851639
1,101,homogeneous_anchor,0.626325,0.944988,6371.940112


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,4class,4class_parity_language,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6


In [8]:
# Train or reuse seed pair 202
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(202)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 202].reset_index(drop=True))


Seed 202 :: reusing existing homogeneous_anchor checkpoint...

Seed 202 :: reusing existing heterogeneous_sampled checkpoint...


,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,202,heterogeneous_sampled,0.564488,1.090982,4149.596533
1,202,homogeneous_anchor,0.598940,1.063645,17263.351055


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,202,4class,4class_parity_language,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.48047,28,1,True,6


In [9]:
# Train or reuse seed pair 210
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(210)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 210].reset_index(drop=True))


Seed 210 :: reusing existing homogeneous_anchor checkpoint...

Seed 210 :: reusing existing heterogeneous_sampled checkpoint...


,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,210,heterogeneous_sampled,0.564046,1.173871,36700.814634
1,210,homogeneous_anchor,0.546378,1.259035,4098.420524


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,210,4class,4class_parity_language,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,31,1,True,6


In [10]:
# Train or reuse seed pair 340
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(340)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 340].reset_index(drop=True))


Seed 340 :: reusing existing homogeneous_anchor checkpoint...

Seed 340 :: reusing existing heterogeneous_sampled checkpoint...


,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,340,heterogeneous_sampled,0.562721,1.101743,6617.713672
1,340,homogeneous_anchor,0.534452,1.316056,4092.126215


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,340,4class,4class_parity_language,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,31,1,True,6


In [11]:
# Train or reuse seed pair 440
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(440)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 440].reset_index(drop=True))


Seed 440 :: reusing existing homogeneous_anchor checkpoint...

Seed 440 :: training heterogeneous_sampled...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.315  test_acc=0.414  (2.8 min)
  epoch=002  train_acc=0.422  test_acc=0.412  (2.7 min)
  epoch=003  train_acc=0.461  test_acc=0.454  (2.7 min)
  epoch=004  train_acc=0.476  test_acc=0.474  (2.7 min)
  epoch=005  train_acc=0.479  test_acc=0.478  (2.6 min)
  epoch=006  train_acc=0.488  test_acc=0.477  (2.7 min)
  epoch=007  train_acc=0.495  test_acc=0.447  (2.7 min)
  epoch=008  train_acc=0.478  test_acc=0.452  (2.7 min)
  epoch=009  train_acc=0.500  test_acc=0.473  (2.6 min)
  epoch=010  train_acc=0.495  test_acc=0.462  (2.7 min)
  epoch=011  train_acc=0.499  test_acc=0.447  (2.8 min)
  epoch=012  train_acc=0.501  test_acc=0.450  (2.8 min)
  epoch=013  train_acc=0.511  test_acc=0.460  (3.5 min)
  epoch=014  train_acc=0.507  test_acc=0.462  (3.6 min)
         LR reduced: 4.00e-03 → 2.00e-03
  epoch=015  train_acc=0.510  test_acc=0.473  (4.0 min)
  epoch=016  train_acc=0.531  test_acc=0.474  (3.8 min)
  epoch=017  train_acc=0.532  test_acc=0.451  (3.7 min)
  epoch

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,440,heterogeneous_sampled,0.451855,1.134629,8411.823550
1,440,homogeneous_anchor,0.587456,0.952413,4226.259442


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,440,4class,4class_parity_language,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.91595,28,1,True,6


In [12]:
# Final status summary
pair_df, checkpoint_df, paired_acc_df = show_run_status()

if pair_df.empty:
    print("No saved seed summaries yet.")
else:
    display(pair_df)

if checkpoint_df.empty:
    print("No saved checkpoint summaries yet.")
else:
    display(checkpoint_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss"]])

if not paired_acc_df.empty:
    display(paired_acc_df)
    print(f"Saved paired accuracy view to: {PAIRED_ACC_CSV}")

,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,4class,4class_parity_language,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6
1,202,4class,4class_parity_language,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.480470,28,1,True,6
2,210,4class,4class_parity_language,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,31,1,True,6
3,340,4class,4class_parity_language,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,31,1,True,6
4,440,4class,4class_parity_language,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.915950,28,1,True,6


,pair_seed,pair_role,final_test_acc,final_test_loss
0,101,heterogeneous_sampled,0.494700,1.190044
1,101,homogeneous_anchor,0.626325,0.944988
2,202,heterogeneous_sampled,0.564488,1.090982
3,202,homogeneous_anchor,0.598940,1.063645
4,210,heterogeneous_sampled,0.564046,1.173871
5,210,homogeneous_anchor,0.546378,1.259035
6,340,heterogeneous_sampled,0.562721,1.101743
7,340,homogeneous_anchor,0.534452,1.316056
8,440,heterogeneous_sampled,0.451855,1.134629
9,440,homogeneous_anchor,0.587456,0.952413


pair_role,pair_seed,heterogeneous_sampled,homogeneous_anchor,hetero_minus_hom
0,101,0.494700,0.626325,-0.131625
1,202,0.564488,0.598940,-0.034452
2,210,0.564046,0.546378,0.017668
3,340,0.562721,0.534452,0.028269
4,440,0.451855,0.587456,-0.135601


Saved paired accuracy view to: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1\4class_lognormal\seeded_run_1_paired_accuracy.csv
